# Building a Basic Graph with LangGraph

This notebook demonstrates how to build a complete RAG system using LangGraph, combining all the concepts we've learned:

- **States and Nodes**: Define state structure and node functions
- **QDrant Integration**: Use vector database for retrieval
- **LCEL Chains**: Compose LangChain components
- **Graph Flow**: Connect nodes with edges

## What We'll Build

A sophisticated RAG system that:
1. **Processes** user questions
2. **Retrieves** relevant context from QDrant
3. **Generates** responses using local LLM
4. **Handles** edge cases and errors
5. **Provides** confidence scores


In [15]:
# Import all required libraries
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import List, Optional, Dict, Any
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
import numpy as np

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


## 1. Define the State Structure

First, let's define a comprehensive state that will hold all the information we need throughout our RAG pipeline.


In [16]:
# Define the state structure for our RAG system
class RAGState(TypedDict):
    # Input
    question: str
    
    # Processing steps
    processed_question: Optional[str]
    retrieved_documents: List[Document]
    context: Optional[str]
    
    # Output
    response: Optional[str]
    confidence_score: Optional[float]
    
    # Metadata
    processing_time: Optional[float]
    error_message: Optional[str]
    retrieval_scores: Optional[List[float]]

print("✅ State structure defined!")
print("State fields:", list(RAGState.__annotations__.keys()))


✅ State structure defined!
State fields: ['question', 'processed_question', 'retrieved_documents', 'context', 'response', 'confidence_score', 'processing_time', 'error_message', 'retrieval_scores']


## 2. Set Up Models and Vector Store

Let's initialize our models and set up the QDrant vector store.


In [17]:
# Initialize models
chat_model = ChatOllama(model="gpt-oss:20b", temperature=0.7)
embedding_model = OllamaEmbeddings(model="embeddinggemma:latest")

# Get embedding dimension
test_embedding = embedding_model.embed_query("test")
embedding_dim = len(test_embedding)

# Set up QDrant
client = QdrantClient(":memory:")
collection_name = "ai_knowledge_base"

# Create collection
try:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE
        )
    )
    print(f"✅ QDrant collection '{collection_name}' created!")
except Exception as e:
    print(f"Collection might already exist: {e}")

# Create LangChain vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model
)

print("✅ Models and vector store initialized!")
print(f"Embedding dimension: {embedding_dim}")


✅ QDrant collection 'ai_knowledge_base' created!
✅ Models and vector store initialized!
Embedding dimension: 768


In [18]:
# Add sample documents to the vector store
sample_documents = [
    Document(
        page_content="Artificial Intelligence (AI) is the simulation of human intelligence in machines that are programmed to think and learn like humans.",
        metadata={"topic": "AI", "source": "intro_guide", "difficulty": "beginner"}
    ),
    Document(
        page_content="Machine Learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed.",
        metadata={"topic": "ML", "source": "ml_basics", "difficulty": "intermediate"}
    ),
    Document(
        page_content="Deep Learning uses artificial neural networks with multiple layers to process data and learn complex patterns.",
        metadata={"topic": "Deep Learning", "source": "neural_networks", "difficulty": "advanced"}
    ),
    Document(
        page_content="Natural Language Processing (NLP) is a branch of AI that helps computers understand, interpret, and manipulate human language.",
        metadata={"topic": "NLP", "source": "language_processing", "difficulty": "intermediate"}
    ),
    Document(
        page_content="Computer Vision is a field of AI that trains computers to interpret and understand visual information from the world.",
        metadata={"topic": "CV", "source": "visual_processing", "difficulty": "intermediate"}
    ),
    Document(
        page_content="Vector databases are specialized databases designed to store and search high-dimensional vectors efficiently for similarity search.",
        metadata={"topic": "Vector DB", "source": "database_guide", "difficulty": "advanced"}
    )
]

# Add documents to vector store
vector_store.add_documents(sample_documents)

print(f"✅ Added {len(sample_documents)} documents to vector store")
print("Sample documents:")
for i, doc in enumerate(sample_documents, 1):
    print(f"{i}. {doc.page_content[:60]}... (Topic: {doc.metadata['topic']})")


✅ Added 6 documents to vector store
Sample documents:
1. Artificial Intelligence (AI) is the simulation of human inte... (Topic: AI)
2. Machine Learning is a subset of AI that enables computers to... (Topic: ML)
3. Deep Learning uses artificial neural networks with multiple ... (Topic: Deep Learning)
4. Natural Language Processing (NLP) is a branch of AI that hel... (Topic: NLP)
5. Computer Vision is a field of AI that trains computers to in... (Topic: CV)
6. Vector databases are specialized databases designed to store... (Topic: Vector DB)


## 3. Create Node Functions

Now let's create the individual nodes that will process our state at each step of the pipeline.


In [19]:
# Node 1: Process and validate the question
def process_question(state: RAGState) -> RAGState:
    """Process and validate the user's question."""
    import time
    start_time = time.time()
    
    question = state["question"]
    
    # Basic validation
    if not question or len(question.strip()) < 3:
        return {
            "error_message": "Please provide a more detailed question.",
            "processed_question": question
        }
    
    # Clean and process the question
    processed_question = question.strip().lower()
    
    # Add some basic processing
    if not processed_question.endswith('?'):
        processed_question += '?'
    
    processing_time = time.time() - start_time
    
    return {
        "processed_question": processed_question,
        "processing_time": processing_time,
        "error_message": None
    }

print("✅ Question processing node created!")


✅ Question processing node created!


In [20]:
# Node 2: Retrieve relevant documents
def retrieve_documents(state: RAGState) -> RAGState:
    """Retrieve relevant documents from the vector store."""
    import time
    start_time = time.time()
    
    # Check if there was an error in previous step
    if state.get("error_message"):
        return {"retrieved_documents": [], "retrieval_scores": []}
    
    question = state["processed_question"] or state["question"]
    
    try:
        # Create retriever
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})
        
        # Retrieve documents
        retrieved_docs = retriever.invoke(question)
        
        # Get similarity scores (for demonstration)
        query_embedding = embedding_model.embed_query(question)
        scores = []
        
        for doc in retrieved_docs:
            doc_embedding = embedding_model.embed_query(doc.page_content)
            # Calculate cosine similarity
            similarity = np.dot(query_embedding, doc_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
            )
            scores.append(float(similarity))
        
        retrieval_time = time.time() - start_time
        
        return {
            "retrieved_documents": retrieved_docs,
            "retrieval_scores": scores,
            "processing_time": state.get("processing_time", 0) + retrieval_time
        }
        
    except Exception as e:
        return {
            "retrieved_documents": [],
            "retrieval_scores": [],
            "error_message": f"Retrieval error: {str(e)}"
        }

print("✅ Document retrieval node created!")


✅ Document retrieval node created!


In [21]:
# Node 3: Generate response
def generate_response(state: RAGState) -> RAGState:
    """Generate a response based on the retrieved context."""
    import time
    start_time = time.time()
    
    # Check if there was an error in previous steps
    if state.get("error_message"):
        return {"response": f"Error: {state['error_message']}"}
    
    question = state["processed_question"] or state["question"]
    retrieved_docs = state.get("retrieved_documents", [])
    
    if not retrieved_docs:
        return {
            "response": "I don't have enough information to answer your question. Please try rephrasing it.",
            "confidence_score": 0.0
        }
    
    try:
        # Create context from retrieved documents
        context = "\\n".join([doc.page_content for doc in retrieved_docs])
        
        # Create RAG prompt
        rag_prompt = ChatPromptTemplate.from_template("""
        Context: {context}
        
        Question: {question}
        
        Answer the question based on the provided context. If the context doesn't contain enough information, say so.
        Be concise and accurate.
        """)
        
        # Create chain
        rag_chain = rag_prompt | chat_model | StrOutputParser()
        
        # Generate response
        response = rag_chain.invoke({"context": context, "question": question})
        
        # Calculate confidence score based on retrieval scores
        scores = state.get("retrieval_scores", [])
        confidence = np.mean(scores) if scores else 0.0
        
        generation_time = time.time() - start_time
        
        return {
            "response": response,
            "confidence_score": float(confidence),
            "context": context,
            "processing_time": state.get("processing_time", 0) + generation_time
        }
        
    except Exception as e:
        return {
            "response": f"Error generating response: {str(e)}",
            "confidence_score": 0.0,
            "error_message": f"Generation error: {str(e)}"
        }

print("✅ Response generation node created!")


✅ Response generation node created!


## 4. Build the Graph

Now let's put all our nodes together and create the complete graph.


In [22]:
# Create the graph
def create_rag_graph():
    """Create a complete RAG graph with all nodes."""
    
    # Create the graph builder
    graph_builder = StateGraph(RAGState)
    
    # Add nodes
    graph_builder.add_node("process_question", process_question)
    graph_builder.add_node("retrieve_documents", retrieve_documents)
    graph_builder.add_node("generate_response", generate_response)
    
    # Define the flow
    graph_builder.add_edge(START, "process_question")
    graph_builder.add_edge("process_question", "retrieve_documents")
    graph_builder.add_edge("retrieve_documents", "generate_response")
    graph_builder.add_edge("generate_response", END)
    
    # Compile the graph
    return graph_builder.compile()

# Create the graph
rag_graph = create_rag_graph()

print("✅ RAG graph created successfully!")
print("Graph structure:")
print(rag_graph)


✅ RAG graph created successfully!
Graph structure:


## 5. Test the Graph

Let's test our complete RAG system with various questions to see how it performs.


In [23]:
# Test the graph with different questions
test_questions = [
    "What is machine learning?",
    "How does deep learning work?",
    "What is the difference between AI and ML?",
    "What are vector databases used for?",
    "Tell me about computer vision",
    "What is natural language processing?"
]

print("Testing RAG Graph:")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\\n{i}. Question: {question}")
    print("-" * 50)
    
    # Run the graph
    result = rag_graph.invoke({"question": question})
    
    # Display results
    print(f"Response: {result['response']}")
    print(f"Confidence Score: {result.get('confidence_score', 0):.3f}")
    print(f"Processing Time: {result.get('processing_time', 0):.2f}s")
    
    if result.get('retrieved_documents'):
        print(f"Retrieved {len(result['retrieved_documents'])} documents:")
        for j, doc in enumerate(result['retrieved_documents'], 1):
            print(f"  {j}. {doc.page_content[:60]}... (Topic: {doc.metadata['topic']})")
    
    if result.get('error_message'):
        print(f"Error: {result['error_message']}")
    
    print("\\n" + "="*60)


Testing RAG Graph:
\n1. Question: What is machine learning?
--------------------------------------------------
Response: Machine learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed.
Confidence Score: 0.749
Processing Time: 21.66s
Retrieved 3 documents:
  1. Deep Learning uses artificial neural networks with multiple ... (Topic: Deep Learning)
  2. Machine Learning is a subset of AI that enables computers to... (Topic: ML)
  3. Computer Vision is a field of AI that trains computers to in... (Topic: CV)
\n============================================================
\n2. Question: How does deep learning work?
--------------------------------------------------
Response: Deep learning works by training multi‑layer artificial neural networks to learn complex patterns from data. Each layer transforms its input into higher‑level features; the network adjusts its weights through back‑propagation and gradient descent so that 

## 6. Advanced Features

Let's add some advanced features like error handling and conditional routing.


In [24]:
# Test error handling
print("Testing Error Handling:")
print("=" * 40)

# Test with empty question
result = rag_graph.invoke({"question": ""})
print(f"Empty question result: {result.get('error_message', 'No error')}")

# Test with very short question
result = rag_graph.invoke({"question": "Hi"})
print(f"Short question result: {result.get('error_message', 'No error')}")

# Test with normal question
result = rag_graph.invoke({"question": "What is AI?"})
print(f"Normal question - Response: {result['response'][:100]}...")
print(f"Confidence: {result.get('confidence_score', 0):.3f}")

print("\\n✅ Error handling test completed!")


Testing Error Handling:
Empty question result: Please provide a more detailed question.
Short question result: Please provide a more detailed question.
Normal question - Response: Artificial Intelligence (AI) is the field of computer science that develops systems capable of perfo...
Confidence: 0.741
\n✅ Error handling test completed!


## 7. Summary

Congratulations! You've successfully built a complete RAG system using LangGraph. Here's what we accomplished:

### ✅ What We Built:
1. **State Management**: Defined a comprehensive state structure
2. **Node Functions**: Created processing, retrieval, and generation nodes
3. **Graph Structure**: Connected nodes with proper flow
4. **Error Handling**: Added robust error handling throughout
5. **Performance Metrics**: Tracked processing time and confidence scores

### 🔧 Key Features:
- **Vector Search**: Uses QDrant for efficient similarity search
- **Local LLM**: Runs entirely on your machine with Ollama
- **Error Handling**: Graceful handling of edge cases
- **Performance Tracking**: Monitors processing time and confidence
- **Modular Design**: Easy to extend and modify

### 🚀 Next Steps:
- Add more sophisticated routing logic
- Implement conversation memory
- Add streaming responses
- Create a web interface
- Deploy to production

You now have a solid foundation for building production RAG applications with LangGraph!
